# Model 1 Performance — Random Forest Classifier

## Performance of Model 1

This notebook evaluates the performance of Model 1, a Random Forest
Classifier, for the STADIOalot return-prediction problem.

### Target Variable

`Return_Status`

- `0` = Not Returned
- `1` = Returned

### Performance Measures

The following measures are calculated:

- Accuracy
- Precision
- Recall
- F1-Score
- ROC-AUC
- Confusion Matrix
- Classification Report
- Bootstrap 95% Confidence Intervals

### Statistical Analysis

Bootstrap resampling with 1,000 iterations is used to estimate 95% confidence intervals for the performance metrics.

### Input Dataset

`data/processed/returns_features.csv`

### Trained Model

`models/model1_random_forest.joblib`

### Outputs

`experimental_results/model1_bootstrap_confidence_intervals.csv`

In [17]:
# ============================================================
# 1. IMPORT LIBRARIES AND DEFINE PROJECT PATHS
# ============================================================

from pathlib import Path
import warnings
import joblib

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

warnings.filterwarnings("ignore")

# ============================================================
# PANDAS DISPLAY SETTINGS
# ============================================================

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.4f}".format)

# ============================================================
# FIND PROJECT ROOT
# ============================================================

ROOT = Path.cwd()

while ROOT != ROOT.parent and not (ROOT / "data").exists():
    ROOT = ROOT.parent

ROOT = ROOT.resolve()

# ============================================================
# DEFINE PROJECT PATHS
# ============================================================

DATA_PATH = (
    ROOT
    / "data"
    / "processed"
    / "returns_features.csv"
)

MODEL_PATH = (
    ROOT
    / "models"
    / "model1_random_forest.joblib"
)

RESULTS_DIR = (
    ROOT
    / "experimental_results"
)

METRICS_PATH = (
    RESULTS_DIR
    / "model1_random_forest_metrics.csv"
)

BOOTSTRAP_PATH = (
    RESULTS_DIR
    / "model1_bootstrap_confidence_intervals.csv"
)

# ============================================================
# CREATE RESULTS DIRECTORY IF NECESSARY
# ============================================================

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ============================================================
# DISPLAY PATHS
# ============================================================

path_table = pd.DataFrame({
    "File": [
        "Feature Engineered Dataset",
        "Model 1 Performance Metrics",
        "Model 1 Bootstrap Confidence Intervals"
    ],
    "Full Path": [
        str(DATA_PATH.resolve()),
        str(METRICS_PATH.resolve()),
        str(BOOTSTRAP_PATH.resolve())
    ]
})

display(path_table)

,File,Full Path
0,Feature Engineered Dataset,C:\Users\payal\OneDrive - STADIO\2026 Post Graduate Diploma Semester 3\Capstone Project\Assignment\STADIOalot-Return-Prediction\data\processed\returns_features.csv
1,Model 1 Performance Metrics,C:\Users\payal\OneDrive - STADIO\2026 Post Graduate Diploma Semester 3\Capstone Project\Assignment\STADIOalot-Return-Prediction\experimental_results\model1_random_forest_metrics.csv
2,Model 1 Bootstrap Confidence Intervals,C:\Users\payal\OneDrive - STADIO\2026 Post Graduate Diploma Semester 3\Capstone Project\Assignment\STADIOalot-Return-Prediction\experimental_results\model1_bootstrap_confidence_intervals.csv


In [18]:
# ============================================================
# 2. LOAD FEATURE-ENGINEERED DATASET
# ============================================================

TARGET_COLUMN = "Return_Status"

# ------------------------------------------------------------
# CHECK DATASET
# ------------------------------------------------------------

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found:\n{DATA_PATH}"
    )

# ------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------

df = pd.read_csv(
    DATA_PATH
)

# ------------------------------------------------------------
# VALIDATE TARGET
# ------------------------------------------------------------

if TARGET_COLUMN not in df.columns:
    raise ValueError(
        f"Target column '{TARGET_COLUMN}' was not found."
    )

if df[TARGET_COLUMN].isna().any():
    raise ValueError(
        f"Target column '{TARGET_COLUMN}' contains missing values."
    )

# ------------------------------------------------------------
# SEPARATE FEATURES AND TARGET
# ------------------------------------------------------------

X = df.drop(
    columns=[TARGET_COLUMN]
)

y = df[TARGET_COLUMN].astype(int)

# ------------------------------------------------------------
# DATASET SUMMARY
# ------------------------------------------------------------

dataset_summary = pd.DataFrame({
    "Dataset Component": [
        "Dataset File",
        "Rows",
        "Columns",
        "Predictor Features",
        "Target Variable",
        "Target Encoding"
    ],
    "Value": [
        str(DATA_PATH.resolve()),
        df.shape[0],
        df.shape[1],
        X.shape[1],
        TARGET_COLUMN,
        "0 = Not Returned; 1 = Returned"
    ]
})

display(dataset_summary)

# ============================================================
# LOAD DATASET
# ============================================================

df = pd.read_csv(DATA_PATH)

print("First Five rows of the dataset")
display(df.head())

,Dataset Component,Value
0,Dataset File,C:\Users\payal\OneDrive - STADIO\2026 Post Graduate Diploma Semester 3\Capstone Project\Assignment\STADIOalot-Return-Prediction\data\processed\returns_features.csv
1,Rows,10000
2,Columns,20
3,Predictor Features,19
4,Target Variable,Return_Status
5,Target Encoding,0 = Not Returned; 1 = Returned


First Five rows of the dataset


,Product_Category,Product_Price,Order_Quantity,Return_Status,User_Age,User_Gender,User_Location,Payment_Method,Shipping_Method,Discount_Applied,order_year,order_month,order_day,order_day_of_week,order_week_of_year,order_is_weekend,order_gross_value,discount_fraction,discount_amount_est,age_band
0,Clothing,411.5900,3,1,58,Male,City54,Debit Card,Next-Day,45.2700,2023,8,5,5,31,1,1234.7700,0.4527,186.3268,55-64
1,Books,288.8800,3,1,68,Female,City85,Credit Card,Express,47.7900,2023,10,9,0,41,0,866.6400,0.4779,138.0558,65+
2,Toys,390.0300,5,0,22,Female,City30,Debit Card,Next-Day,26.6400,2023,5,6,5,18,1,1950.1500,0.2664,103.9040,18-24
3,Toys,401.0900,3,0,40,Male,City95,PayPal,Next-Day,15.3700,2024,8,29,3,35,0,1203.2700,0.1537,61.6475,35-44
4,Books,110.0900,4,0,34,Female,City80,Gift Card,Standard,16.3700,2023,1,16,0,3,0,440.3600,0.1637,18.0217,25-34


In [19]:
# ============================================================
# 3. TARGET CLASS DISTRIBUTION
# ============================================================

target_distribution = (
    y
    .value_counts()
    .sort_index()
    .rename_axis("Return_Status")
    .reset_index(name="Frequency")
)

target_distribution["Class"] = (
    target_distribution["Return_Status"]
    .map({
        0: "Not Returned",
        1: "Returned"
    })
)

target_distribution["Percentage"] = (
    target_distribution["Frequency"]
    / len(y)
    * 100
)

target_distribution = target_distribution[
    [
        "Return_Status",
        "Class",
        "Frequency",
        "Percentage"
    ]
]

display(
    target_distribution
)

,Return_Status,Class,Frequency,Percentage
0,0,Not Returned,4948,49.4800
1,1,Returned,5052,50.5200


In [20]:
# ============================================================
# 4. RECREATE MODEL 1 TRAIN / TEST SPLIT
# ============================================================

TEST_SIZE = 0.20
RANDOM_STATE = 42

# ------------------------------------------------------------
# STRATIFIED 80/20 SPLIT
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

# ------------------------------------------------------------
# SPLIT SUMMARY
# ------------------------------------------------------------

split_summary = pd.DataFrame({
    "Dataset Portion": [
        "Full Dataset",
        "Training Set",
        "Testing Set"
    ],
    "Rows": [
        len(X),
        len(X_train),
        len(X_test)
    ],
    "Percentage": [
        100.00,
        len(X_train) / len(X) * 100,
        len(X_test) / len(X) * 100
    ]
})

print("Split Summary")
display(
    split_summary
)
# ============================================================
# 5. VERIFY TARGET DISTRIBUTION AFTER SPLIT
# ============================================================

split_target_report = pd.DataFrame({
    "Return Status": [
        "Not Returned",
        "Returned"
    ],
    "Class": [
        0,
        1
    ],
    "Training Count": [
        int((y_train == 0).sum()),
        int((y_train == 1).sum())
    ],
    "Training Percentage": [
        (y_train == 0).mean() * 100,
        (y_train == 1).mean() * 100
    ],
    "Testing Count": [
        int((y_test == 0).sum()),
        int((y_test == 1).sum())
    ],
    "Testing Percentage": [
        (y_test == 0).mean() * 100,
        (y_test == 1).mean() * 100
    ]
})

print("Display Split Target Report")
display(split_target_report)

Split Summary


,Dataset Portion,Rows,Percentage
0,Full Dataset,10000,100.0000
1,Training Set,8000,80.0000
2,Testing Set,2000,20.0000


Display Split Target Report


,Return Status,Class,Training Count,Training Percentage,Testing Count,Testing Percentage
0,Not Returned,0,3958,49.4750,990,49.5000
1,Returned,1,4042,50.5250,1010,50.5000


In [21]:
# ============================================================
# 6. LOAD SAVED MODEL 1 PIPELINE
# ============================================================

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Model 1 was not found at:\n{MODEL_PATH}\n\n"
        "Run the Model 1 training notebook first."
    )

# ------------------------------------------------------------
# LOAD SAVED PIPELINE
# ------------------------------------------------------------

model1 = joblib.load(
    MODEL_PATH
)

# ------------------------------------------------------------
# VALIDATE LOADED MODEL
# ------------------------------------------------------------

if not hasattr(model1, "predict"):
    raise TypeError(
        "Loaded Model 1 does not have a predict() method."
    )

if not hasattr(model1, "predict_proba"):
    raise TypeError(
        "Loaded Model 1 does not have a predict_proba() method."
    )

# ------------------------------------------------------------
# MODEL VALIDATION TABLE
# ------------------------------------------------------------

model_validation = pd.DataFrame({
    "Validation Check": [
        "Model file exists",
        "Model loaded",
        "Predict method available",
        "Predict probability method available"
    ],
    "Result": [
        MODEL_PATH.exists(),
        model1 is not None,
        hasattr(model1, "predict"),
        hasattr(model1, "predict_proba")
    ]
})

display(
    model_validation
)

,Validation Check,Result
0,Model file exists,True
1,Model loaded,True
2,Predict method available,True
3,Predict probability method available,True


In [22]:
# ============================================================
# 7. GENERATE MODEL 1 PREDICTIONS
# ============================================================

# ------------------------------------------------------------
# CLASS PREDICTIONS
# ------------------------------------------------------------

y_pred = model1.predict(
    X_test
)

# ------------------------------------------------------------
# PROBABILITY PREDICTIONS
# ------------------------------------------------------------

y_probability = (
    model1
    .predict_proba(X_test)[:, 1]
)

# ------------------------------------------------------------
# VALIDATE PREDICTION LENGTH
# ------------------------------------------------------------

if len(y_pred) != len(y_test):
    raise ValueError(
        "Number of predictions does not match test-set size."
    )

if len(y_probability) != len(y_test):
    raise ValueError(
        "Number of probability predictions does not match test-set size."
    )

# ------------------------------------------------------------
# PREDICTION SUMMARY
# ------------------------------------------------------------

prediction_summary = pd.DataFrame({
    "Output": [
        "Class Predictions",
        "Return Probabilities"
    ],
    "Number of Predictions": [
        len(y_pred),
        len(y_probability)
    ],
    "Description": [
        "0 = Not Returned; 1 = Returned",
        "Probability of Returned class"
    ]
})

print("Prediction Summary")
display(
    prediction_summary
)

# ============================================================
# PREDICTION PREVIEW TABLE
# ============================================================

prediction_preview = pd.DataFrame({
    "Actual Return Status": y_test.reset_index(drop=True),
    "Predicted Return Status": pd.Series(y_pred),
    "Return Probability": pd.Series(y_probability)
})

prediction_preview["Actual Class"] = (
    prediction_preview["Actual Return Status"]
    .map({
        0: "Not Returned",
        1: "Returned"
    })
)

prediction_preview["Predicted Class"] = (
    prediction_preview["Predicted Return Status"]
    .map({
        0: "Not Returned",
        1: "Returned"
    })
)

prediction_preview = prediction_preview[
    [
        "Actual Return Status",
        "Actual Class",
        "Predicted Return Status",
        "Predicted Class",
        "Return Probability"
    ]
]

print("Prediction Preview")
display(
    prediction_preview.head(10)
)

Prediction Summary


,Output,Number of Predictions,Description
0,Class Predictions,2000,0 = Not Returned; 1 = Returned
1,Return Probabilities,2000,Probability of Returned class


Prediction Preview


,Actual Return Status,Actual Class,Predicted Return Status,Predicted Class,Return Probability
0,0,Not Returned,0,Not Returned,0.4977
1,1,Returned,1,Returned,0.5195
2,0,Not Returned,0,Not Returned,0.4701
3,1,Returned,0,Not Returned,0.4717
4,0,Not Returned,1,Returned,0.5230
5,0,Not Returned,0,Not Returned,0.4831
6,0,Not Returned,1,Returned,0.5292
7,1,Returned,0,Not Returned,0.4837
8,1,Returned,0,Not Returned,0.4196
9,1,Returned,0,Not Returned,0.4818


In [23]:
# ============================================================
# 8. CALCULATE MODEL 1 PERFORMANCE METRICS
# ============================================================

accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    zero_division=0
)

roc_auc = roc_auc_score(
    y_test,
    y_probability
)

# ============================================================
# STRUCTURED METRICS TABLE
# ============================================================

metrics_table = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-Score",
        "ROC-AUC"
    ],

    "Score": [
        accuracy,
        precision,
        recall,
        f1,
        roc_auc
    ],

    "Purpose": [
        "Overall proportion of correct predictions",
        "Proportion of predicted returns that were actually returned",
        "Proportion of actual returns correctly identified",
        "Harmonic balance between precision and recall",
        "Ability to discriminate between returned and non-returned orders"
    ]
})

print("Performance Metrics")
display(
    metrics_table
)

Performance Metrics


,Metric,Score,Purpose
0,Accuracy,0.5000,Overall proportion of correct predictions
1,Precision,0.5048,Proportion of predicted returns that were actually returned
2,Recall,0.5198,Proportion of actual returns correctly identified
3,F1-Score,0.5122,Harmonic balance between precision and recall
4,ROC-AUC,0.4925,Ability to discriminate between returned and non-returned orders


In [24]:
# ============================================================
# 9. CLASSIFICATION REPORT
# ============================================================

classification_report_dict = classification_report(
    y_test,
    y_pred,
    output_dict=True,
    zero_division=0
)

classification_table = (
    pd.DataFrame(
        classification_report_dict
    )
    .transpose()
    .reset_index()
    .rename(
        columns={
            "index": "Class"
        }
    )
)

classification_table["Class"] = (
    classification_table["Class"]
    .replace({
        "0": "Not Returned",
        "1": "Returned",
        "macro avg": "Macro Average",
        "weighted avg": "Weighted Average",
        "accuracy": "Accuracy"
    })
)

print("Classification Table")
display(
    classification_table
)

Classification Table


,Class,precision,recall,f1-score,support
0,Not Returned,0.4948,0.4798,0.4872,990.0000
1,Returned,0.5048,0.5198,0.5122,1010.0000
2,Accuracy,0.5000,0.5000,0.5000,0.5000
3,Macro Average,0.4998,0.4998,0.4997,2000.0000
4,Weighted Average,0.4998,0.5000,0.4998,2000.0000


In [27]:
# ============================================================
# 10. CONFUSION MATRIX INTERPRETATION
# ============================================================

cm = confusion_matrix(
    y_test,
    y_pred
)

tn, fp, fn, tp = cm.ravel()

confusion_summary = pd.DataFrame({
    "Measure": [
        "True Negatives (TN)",
        "False Positives (FP)",
        "False Negatives (FN)",
        "True Positives (TP)"
    ],

    "Count": [
        tn,
        fp,
        fn,
        tp
    ],

    "Interpretation": [
        "Correctly predicted orders that were not returned",
        "Orders predicted as returned but actually not returned",
        "Orders predicted as not returned but actually returned",
        "Correctly predicted returned orders"
    ]
})

print("Confusion Matrix Summary")
display(
    confusion_summary
)

Confusion Matrix Summary


,Measure,Count,Interpretation
0,True Negatives (TN),475,Correctly predicted orders that were not returned
1,False Positives (FP),515,Orders predicted as returned but actually not returned
2,False Negatives (FN),485,Orders predicted as not returned but actually returned
3,True Positives (TP),525,Correctly predicted returned orders


In [28]:
# ============================================================
# 11. BOOTSTRAP 95% CONFIDENCE INTERVALS
# ============================================================

"""
Purpose:
    Estimate 95% bootstrap confidence intervals for the Model 1
    XGBoost performance metrics.

Important:
    The Model 1 performance metrics CSV was already created in Part B.

    Therefore, this Part C notebook does NOT save:
        experimental_results/model1_random_forest_metrics.csv

    This section saves ONLY:
        experimental_results/model1_bootstrap_confidence_intervals.csv
"""

# ------------------------------------------------------------
# Bootstrap configuration
# ------------------------------------------------------------
N_BOOTSTRAPS = 1000
n_test = len(y_test)
BOOTSTRAP_RANDOM_STATE = 42
rng = np.random.default_rng(
    BOOTSTRAP_RANDOM_STATE
)
# ------------------------------------------------------------
# Create storage for bootstrap results
# ------------------------------------------------------------

bootstrap_results = {
    "Accuracy": [],
    "Precision": [],
    "Recall": [],
    "F1-Score": [],
    "ROC-AUC": []
}

# ------------------------------------------------------------
# Bootstrap resampling
# ------------------------------------------------------------

for _ in range(N_BOOTSTRAPS):

    indices = rng.integers(
        0,
        n_test,
        n_test
    )

    y_true_bootstrap = y_test.iloc[indices]
    y_pred_bootstrap = np.asarray(y_pred)[indices]
    y_probability_bootstrap = np.asarray(
        y_probability
    )[indices]


    # --------------------------------------------------------
    # Accuracy
    # --------------------------------------------------------

    bootstrap_results["Accuracy"].append(
        accuracy_score(
            y_true_bootstrap,
            y_pred_bootstrap
        )
    )

    # --------------------------------------------------------
    # Precision
    # --------------------------------------------------------

    bootstrap_results["Precision"].append(
        precision_score(
            y_true_bootstrap,
            y_pred_bootstrap,
            zero_division=0
        )
    )
    
    # --------------------------------------------------------
    # Recall
    # --------------------------------------------------------

    bootstrap_results["Recall"].append(
        recall_score(
            y_true_bootstrap,
            y_pred_bootstrap,
            zero_division=0
        )
    )

    # --------------------------------------------------------
    # F1-Score
    # --------------------------------------------------------

    bootstrap_results["F1-Score"].append(
        f1_score(
            y_true_bootstrap,
            y_pred_bootstrap,
            zero_division=0
        )
    )

    # --------------------------------------------------------
    # ROC-AUC
    # --------------------------------------------------------

    # ROC-AUC requires both classes to be present.
    # If a bootstrap sample contains only one class,
    # that iteration is skipped for ROC-AUC.

    if len(
        np.unique(y_true_bootstrap)
    ) == 2:

        bootstrap_results["ROC-AUC"].append(
            roc_auc_score(
                y_true_bootstrap,
                y_probability_bootstrap
            )
        )

# ------------------------------------------------------------
# Observed Model 1 scores
# ------------------------------------------------------------

observed_scores = {
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1-Score": f1,
    "ROC-AUC": roc_auc
}

# ------------------------------------------------------------
# Calculate 95% confidence intervals
# ------------------------------------------------------------

bootstrap_rows = []

for metric, values in bootstrap_results.items():

    bootstrap_rows.append({
        "Model": "Model 1",
        "Algorithm": "Random Forest",
        "Metric": metric,
        "Observed Score": observed_scores[metric],
        "95% CI Lower": np.percentile(
            values,
            2.5
        ),
        "95% CI Upper": np.percentile(
            values,
            97.5
        ),
        "Bootstrap Samples": len(values),
        "Bootstrap Iterations Requested": N_BOOTSTRAPS,
        "Random State": BOOTSTRAP_RANDOM_STATE
    })

bootstrap_ci_table = pd.DataFrame(
    bootstrap_rows
)

# ------------------------------------------------------------
# Display bootstrap confidence intervals
# ------------------------------------------------------------

display(
    bootstrap_ci_table
)

,Model,Algorithm,Metric,Observed Score,95% CI Lower,95% CI Upper,Bootstrap Samples,Bootstrap Iterations Requested,Random State
0,Model 1,Random Forest,Accuracy,0.5000,0.4795,0.5220,1000,1000,42
1,Model 1,Random Forest,Precision,0.5048,0.4748,0.5376,1000,1000,42
2,Model 1,Random Forest,Recall,0.5198,0.4884,0.5526,1000,1000,42
3,Model 1,Random Forest,F1-Score,0.5122,0.4846,0.5391,1000,1000,42
4,Model 1,Random Forest,ROC-AUC,0.4925,0.4677,0.5171,1000,1000,42


In [29]:
# ============================================================
# 12. SAVE MODEL 1 BOOTSTRAP CONFIDENCE INTERVALS
# ============================================================
"""
Part C statistical output.

This is the ONLY output file saved by this section.

Already created in Part B:
    experimental_results/model1_random_forest_metrics.csv

Created here for Part C:
    experimental_results/model1_bootstrap_confidence_intervals.csv
"""

# ------------------------------------------------------------
# Define Part C bootstrap output path
# ------------------------------------------------------------

BOOTSTRAP_RESULT_PATH = (
    ROOT
    / "experimental_results"
    / "model1_bootstrap_confidence_intervals.csv"
)
# ------------------------------------------------------------
# Create output directory if necessary
# ------------------------------------------------------------

BOOTSTRAP_RESULT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)
# ------------------------------------------------------------
# Save bootstrap confidence intervals
# ------------------------------------------------------------

bootstrap_ci_table.to_csv(
    BOOTSTRAP_RESULT_PATH,
    index=False
)

# ------------------------------------------------------------
# Verify saved file
# ------------------------------------------------------------

bootstrap_save_report = pd.DataFrame({
    "Output": [
        "Model 1 Bootstrap Confidence Intervals"
    ],
    "File Name": [
        BOOTSTRAP_RESULT_PATH.name
    ],
    "Full Path": [
        str(BOOTSTRAP_RESULT_PATH.resolve())
    ],
    "Rows Saved": [
        len(bootstrap_ci_table)
    ],
    "File Exists": [
        BOOTSTRAP_RESULT_PATH.exists()
    ]
})

display(
    bootstrap_save_report
)

,Output,File Name,Full Path,Rows Saved,File Exists
0,Model 1 Bootstrap Confidence Intervals,model1_bootstrap_confidence_intervals.csv,C:\Users\payal\OneDrive - STADIO\2026 Post Graduate Diploma Semester 3\Capstone Project\Assignment\STADIOalot-Return-Prediction\experimental_results\model1_bootstrap_confidence_intervals.csv,5,True


In [ ]:
# ============================================================
# 13. MODEL 1 STATISTICAL OUTPUT SUMMARY
# ============================================================

final_model1_summary = pd.DataFrame({
    "Category": [
        "Model",
        "Algorithm",
        "Dataset",
        "Target",
        "Train/Test Split",
        "Random State",
        "Training Rows",
        "Testing Rows",
        "Accuracy",
        "Precision",
        "Recall",
        "F1-Score",
        "ROC-AUC",
        "True Negatives",
        "False Positives",
        "False Negatives",
        "True Positives",
        "Bootstrap Iterations",
        "Bootstrap Confidence Level",
        "Bootstrap Output"
    ],

    "Value": [
        "Model 1",
        "Random Forest Classifier",
        str(DATA_PATH.resolve()),
        "Return_Status",
        "80% / 20% Stratified",
        42,
        len(X_train),
        len(X_test),
        accuracy,
        precision,
        recall,
        f1,
        roc_auc,
        tn,
        fp,
        fn,
        tp,
        N_BOOTSTRAPS,
        "95%",
        str(BOOTSTRAP_PATH.resolve())
    ]
})

display(
    final_model1_summary
)

,Category,Value
0,Model,Model 1
1,Algorithm,Random Forest Classifier
2,Dataset,C:\Users\payal\OneDrive - STADIO\2026 Post Graduate Diploma Semester 3\Capstone Project\Assignment\STADIOalot-Return-Prediction\data\processed\returns_features.csv
3,Target,Return_Status
4,Train/Test Split,80% / 20% Stratified
5,Random State,42
6,Training Rows,8000
7,Testing Rows,2000
8,Accuracy,0.5000
9,Precision,0.5048
